# MNIST MLP3 Muon microbatch RG spectra

This notebook analyzes the matrix-only checkpoints written by
`rg-mnist-muon-microbatch`.

For every layer and checkpoint it computes the ordinary weight ESD

\[
\lambda_i(W_t)=\sigma_i(W_t)^2.
\]

For every successive pair it also constructs the supported relative-flow map

\[
J_t=W_tW_{t-1}^{+}
\]

in output space for wide matrices, or

\[
J_t=W_{t-1}^{+}W_t
\]

in input space for tall matrices. Its ESD is \(\sigma_i(J_t)^2\).

The third spectrum,

\[
|\log \lambda_i(J_t)|,
\]

drops the trivial identity/orthogonal mode at \(\lambda=1\). Each positive
spectrum is fitted with `powerlaw`, and the fitted exponent \(\alpha\) is
plotted against optimizer step. The horizontal line at \(\alpha=2\) is the
RG power-counting hypothesis, not a fitted constraint.

**Fit convention.** `powerlaw` 2.0 defaults to an upper bound
\(\alpha\leq 3\). This notebook explicitly expands the admissible range to
\(1.01\leq\alpha\leq10\), and records whether a result lands on that expanded
boundary.

**Caveats.** The relative-flow operator is basis dependent; this is a
numerical test of the proposal, not a proof of basis invariance. The
classifier `fc3.weight` has only ten singular values, so its power-law fits are
low-sample and should be treated as qualitative.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import powerlaw

from rg_baselines.muon_microbatch_capture import (
    load_microbatch_checkpoint,
    load_microbatch_index,
    log_flow_deviation,
    matrix_esd_eigenvalues,
    relative_flow_esd_eigenvalues,
)


In [ ]:
# Point this at the output directory created by rg-mnist-muon-microbatch.
RUN_DIR = Path("../results/mnist_mlp3_muon_microbatch")

# Optional limits for exploratory analysis.
MAX_CHECKPOINTS = None   # e.g. 500; None analyzes all saved checkpoints
CHECKPOINT_STRIDE = 1    # 1 means truly successive saved checkpoints
PINV_RTOL = 1e-6
POWERLAW_MIN_POINTS = 8
POWERLAW_ALPHA_RANGE = [1.01, 10.0]
ALPHA_BOUNDARY_ATOL = 1e-3
LOG_DEVIATION_ZERO_TOL = 1e-12

index = load_microbatch_index(RUN_DIR)
if MAX_CHECKPOINTS is not None:
    index = index.iloc[: int(MAX_CHECKPOINTS)].copy()
index = index.iloc[:: int(CHECKPOINT_STRIDE)].reset_index(drop=True)
index.head(), index.tail(), len(index)


In [ ]:
def fit_powerlaw(values, *, min_points=POWERLAW_MIN_POINTS):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values) & (values > 0.0)]
    base = {
        "alpha": np.nan,
        "xmin": np.nan,
        "ks_distance": np.nan,
        "n_tail": 0,
        "n_values": int(values.size),
        "tail_fraction": 0.0,
        "alpha_lower_bound": float(POWERLAW_ALPHA_RANGE[0]),
        "alpha_upper_bound": float(POWERLAW_ALPHA_RANGE[1]),
        "alpha_at_boundary": False,
        "fit_error": "",
    }
    if values.size < int(min_points):
        return {**base, "fit_error": "too_few_values"}

    try:
        # powerlaw 2.x uses parameter_ranges (plural). Its built-in default
        # range for Power_Law is alpha in [0, 3], so expand it explicitly.
        fit = powerlaw.Fit(
            values,
            discrete=False,
            verbose=False,
            parameter_ranges={"alpha": list(POWERLAW_ALPHA_RANGE)},
        )
        alpha = float(fit.power_law.alpha)
        xmin = float(fit.power_law.xmin)
        n_tail = int(np.count_nonzero(values >= xmin))
        at_boundary = bool(
            np.isclose(
                alpha,
                POWERLAW_ALPHA_RANGE[0],
                atol=ALPHA_BOUNDARY_ATOL,
                rtol=0.0,
            )
            or np.isclose(
                alpha,
                POWERLAW_ALPHA_RANGE[1],
                atol=ALPHA_BOUNDARY_ATOL,
                rtol=0.0,
            )
        )
        return {
            **base,
            "alpha": alpha,
            "xmin": xmin,
            "ks_distance": float(fit.power_law.D),
            "n_tail": n_tail,
            "tail_fraction": n_tail / max(values.size, 1),
            "alpha_at_boundary": at_boundary,
        }
    except Exception as exc:
        return {
            **base,
            "fit_error": f"{type(exc).__name__}: {exc}",
        }


In [ ]:
fit_rows = []
esd_records = {}
previous_matrices = None
previous_step = None

for row in index.itertuples(index=False):
    payload = load_microbatch_checkpoint(row.checkpoint_path)
    step = int(payload["global_step"])
    matrices = payload["matrices"]

    for layer_name, matrix in matrices.items():
        weight_eigs = matrix_esd_eigenvalues(matrix)
        esd_records[("weight", step, layer_name)] = weight_eigs
        fit_rows.append({
            "spectrum": "weight",
            "global_step": step,
            "previous_step": np.nan,
            "step_delta": np.nan,
            "epoch": int(payload["epoch"]),
            "layer": layer_name,
            "operator_side": "weight",
            **fit_powerlaw(weight_eigs),
        })

        if previous_matrices is None:
            continue

        flow_eigs, side = relative_flow_esd_eigenvalues(
            previous_matrices[layer_name],
            matrix,
            pinv_rtol=PINV_RTOL,
        )
        esd_records[("relative_flow", step, layer_name)] = flow_eigs
        fit_rows.append({
            "spectrum": "relative_flow",
            "global_step": step,
            "previous_step": int(previous_step),
            "step_delta": step - int(previous_step),
            "epoch": int(payload["epoch"]),
            "layer": layer_name,
            "operator_side": side,
            **fit_powerlaw(flow_eigs),
        })

        log_modes = log_flow_deviation(
            flow_eigs,
            zero_tol=LOG_DEVIATION_ZERO_TOL,
        )
        esd_records[("log_flow_deviation", step, layer_name)] = log_modes
        fit_rows.append({
            "spectrum": "log_flow_deviation",
            "global_step": step,
            "previous_step": int(previous_step),
            "step_delta": step - int(previous_step),
            "epoch": int(payload["epoch"]),
            "layer": layer_name,
            "operator_side": side,
            **fit_powerlaw(log_modes),
        })

    previous_matrices = {
        name: value.detach().clone() for name, value in matrices.items()
    }
    previous_step = step

fits = pd.DataFrame(fit_rows).sort_values(
    ["spectrum", "layer", "global_step"]
).reset_index(drop=True)

fits.to_csv(RUN_DIR / "microbatch_powerlaw_fits.csv", index=False)
np.savez_compressed(
    RUN_DIR / "microbatch_esd_spectra.npz",
    **{
        f"{kind}__step_{step:07d}__{layer.replace('.', '_')}": values
        for (kind, step, layer), values in esd_records.items()
    },
)

fits.head(), fits.shape


In [ ]:
boundary_hits = fits[fits["alpha_at_boundary"].astype(bool)]
print(f"Power-law fit rows: {len(fits):,}")
print(f"Expanded-range boundary hits: {len(boundary_hits):,}")
if len(boundary_hits):
    display(
        boundary_hits[
            [
                "spectrum",
                "global_step",
                "layer",
                "alpha",
                "n_tail",
                "n_values",
                "ks_distance",
            ]
        ].head(30)
    )


In [ ]:
def plot_alpha_vs_step(frame, spectrum):
    selected = frame[frame["spectrum"].eq(spectrum)].copy()
    fig, ax = plt.subplots(figsize=(9, 5.5))
    for layer, group in selected.groupby("layer", sort=True):
        group = group[np.isfinite(group["alpha"])].sort_values("global_step")
        ax.plot(
            group["global_step"],
            group["alpha"],
            marker=".",
            linewidth=1.2,
            label=layer,
        )

        boundary = group[group["alpha_at_boundary"].astype(bool)]
        if not boundary.empty:
            ax.scatter(
                boundary["global_step"],
                boundary["alpha"],
                marker="x",
                s=28,
            )

    ax.axhline(
        2.0,
        linestyle="--",
        linewidth=1.4,
        label="RG hypothesis α=2",
    )
    ax.set_xlabel("optimizer step")
    ax.set_ylabel("power-law exponent α")
    ax.set_title(f"MNIST MLP3 Muon: {spectrum} α versus step")
    ax.grid(True, alpha=0.25)
    ax.legend()
    fig.tight_layout()
    return fig, ax

plot_alpha_vs_step(fits, "weight")
plt.show()


In [ ]:
plot_alpha_vs_step(fits, "relative_flow")
plt.show()


In [ ]:
plot_alpha_vs_step(fits, "log_flow_deviation")
plt.show()


In [ ]:
# Inspect ESDs at the first, middle, and last available steps.
def selected_steps_for(spectrum, layer):
    steps = sorted(
        step
        for kind, step, name in esd_records
        if kind == spectrum and name == layer
    )
    if not steps:
        return []
    return sorted(set([steps[0], steps[len(steps) // 2], steps[-1]]))


def plot_selected_esds(spectrum, layer, bins=30):
    fig, ax = plt.subplots(figsize=(8, 5.5))
    for step in selected_steps_for(spectrum, layer):
        values = np.asarray(esd_records[(spectrum, step, layer)], dtype=float)
        values = values[np.isfinite(values) & (values > 0.0)]
        if values.size < 2 or values.min() == values.max():
            continue
        edges = np.logspace(
            np.log10(values.min()),
            np.log10(values.max()),
            bins,
        )
        density, edges = np.histogram(values, bins=edges, density=True)
        centers = np.sqrt(edges[:-1] * edges[1:])
        mask = density > 0.0
        ax.loglog(
            centers[mask],
            density[mask],
            marker="o",
            label=f"step {step}",
        )
    ax.set_xlabel("eigenvalue / mode magnitude")
    ax.set_ylabel("ESD density")
    ax.set_title(f"{spectrum}: {layer}")
    ax.grid(True, which="both", alpha=0.25)
    ax.legend()
    fig.tight_layout()
    return fig, ax

plot_selected_esds("weight", "fc1.weight")
plt.show()


In [ ]:
plot_selected_esds("relative_flow", "fc1.weight")
plt.show()


In [ ]:
plot_selected_esds("log_flow_deviation", "fc1.weight")
plt.show()


In [ ]:
# Compact summary over the final 25 fitted checkpoints.
summary = (
    fits[np.isfinite(fits["alpha"])]
    .sort_values("global_step")
    .groupby(["spectrum", "layer"], as_index=False)
    .tail(25)
    .groupby(["spectrum", "layer"], as_index=False)
    .agg(
        alpha_median=("alpha", "median"),
        alpha_mean=("alpha", "mean"),
        alpha_std=("alpha", "std"),
        last_step=("global_step", "max"),
        median_tail_size=("n_tail", "median"),
        median_tail_fraction=("tail_fraction", "median"),
        median_ks_distance=("ks_distance", "median"),
        boundary_hits=("alpha_at_boundary", "sum"),
    )
)
summary
